# PostgreSQL Writer

This notebook consumes fraud predictions from Kafka and stores them into PostgreSQL.

Pipeline

Kafka
↓

fraud-predictions

↓

Spark Structured Streaming

↓

PostgreSQL

## Imports

In [1]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

import psycopg2
import pandas as pd

## PostgreSQL Configuration

In [2]:
POSTGRES_CONFIG = {

    "host": "localhost",

    "port": 5432,

    "database": "fraud_detection",

    "user": "postgres",

    "password": "Az@7306423517"

}

## Test PostgreSQL Connection

In [3]:
try:

    conn = psycopg2.connect(**POSTGRES_CONFIG)

    print("✅ Connected to PostgreSQL")

    conn.close()

except Exception as e:

    print(e)

✅ Connected to PostgreSQL


## Create Spark Session

In [4]:
spark = (

    SparkSession.builder

    .appName("PostgreSQLWriter")

    .config(
        "spark.jars.packages",

        ",".join([

            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0",

            "org.postgresql:postgresql:42.7.4"

        ])

    )

    .getOrCreate()

)

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/12 12:59:19 WARN Utils: Your hostname, AHMADs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.167 instead (on interface en0)
26/07/12 12:59:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/yaseen/.ivy2.5.2/cache
The jars for the packages stored in: /Users/yaseen/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3bd09ff4-7d13-4449-b35b-4f5f2a0ac402;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.0 in central
	found org.apache.spark#spark

## Stop Previous Streams

In [5]:
for stream in spark.streams.active:

    stream.stop()

print("Previous streams stopped.")

Previous streams stopped.


## Read from Kafka

In [6]:
kafka_df = (

    spark.readStream

    .format("kafka")

    .option(
        "kafka.bootstrap.servers",
        "localhost:9092"
    )

    .option(
        "subscribe",
        "fraud-predictions"
    )

    .option(
        "startingOffsets",
        "latest"
    )

    .load()

)

## Define the Prediction Schema

In [7]:
prediction_schema = StructType([

    StructField("transaction_id", StringType()),

    StructField("event_timestamp", StringType()),

    StructField("Amount", DoubleType()),

    StructField("Prediction", IntegerType()),

    StructField("Fraud_Probability", DoubleType()),

    StructField("Status", StringType()),

    StructField("Risk_Level", StringType())

])

## Parse the JSON

In [8]:
prediction_df = (

    kafka_df

    .selectExpr("CAST(value AS STRING)")

    .select(

        from_json(

            col("value"),

            prediction_schema

        ).alias("data")

    )

    .select("data.*")

)

## Verify Schema

In [9]:
prediction_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Prediction: integer (nullable = true)
 |-- Fraud_Probability: double (nullable = true)
 |-- Status: string (nullable = true)
 |-- Risk_Level: string (nullable = true)



## Preview Stream

In [10]:
preview_query = (

    prediction_df

    .writeStream

    .format("console")

    .outputMode("append")

    .start()

)

# Write Predictions to PostgreSQL

# Write Batch to PostgreSQL

In [11]:
def write_to_postgres(batch_df, batch_id):

    print("\n" + "=" * 70)
    print(f"Writing Batch {batch_id} to PostgreSQL")
    print("=" * 70)

    pdf = batch_df.toPandas()

    if pdf.empty:
        print("No records received.")
        return

    conn = None
    cursor = None

    try:

        conn = psycopg2.connect(**POSTGRES_CONFIG)

        cursor = conn.cursor()

        inserted = 0

        for _, row in pdf.iterrows():

            cursor.execute(
                """
                INSERT INTO fraud_predictions
                (
                    transaction_id,
                    event_timestamp,
                    amount,
                    prediction,
                    fraud_probability,
                    risk_level,
                    status
                )

                VALUES
                (%s,%s,%s,%s,%s,%s,%s)

                ON CONFLICT (transaction_id)
                DO NOTHING;
                """,

                (
                    row["transaction_id"],
                    row["event_timestamp"],
                    float(row["Amount"]),
                    int(row["Prediction"]),
                    float(row["Fraud_Probability"]),
                    row["Risk_Level"],
                    row["Status"]
                )

            )

            inserted += cursor.rowcount

        conn.commit()

        print(f"Inserted {inserted} row(s).")

    except Exception as e:

        print("Database Error")

        print(e)

        if conn:
            conn.rollback()

    finally:

        if cursor:
            cursor.close()

        if conn:
            conn.close()

# Start PostgreSQL Streaming Writer

In [12]:
query = (

    prediction_df

    .writeStream

    .foreachBatch(write_to_postgres)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../checkpoints/postgres_writer"
    )

    .start()

)

## Streaming Status

In [13]:
print(query.status)

{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


## Active Streams

In [14]:
spark.streams.active

## Keep the Stream Running

In [15]:
query.awaitTermination()


Writing Batch 228 to PostgreSQL
-------------------------------------------
Batch: 0
-------------------------------------------
+--------------+---------------+------+----------+-----------------+------+----------+
|transaction_id|event_timestamp|Amount|Prediction|Fraud_Probability|Status|Risk_Level|
+--------------+---------------+------+----------+-----------------+------+----------+
+--------------+---------------+------+----------+-----------------+------+----------+



-------------------------------------------
Batch: 1
-------------------------------------------
Inserted 15 row(s).
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|      transaction_id|     event_timestamp|Amount|Prediction|Fraud_Probability| Status|Risk_Level|
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|7604f08b-e816-4e5...|2026-07-12T07:29:...| 15.99|         0|              0.0|Genuine|       Low|
+--------------------+--------------------+------+----------+-----------------+-------+----------+


Writing Batch 229 to PostgreSQL
Inserted 1 row(s).

Writing Batch 230 to PostgreSQL
-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|      transaction_id|     event_timestamp|Amount|Prediction|Fraud_Probability| Status|Risk_Level|
+------

-------------------------------------------
Batch: 28
-------------------------------------------
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|      transaction_id|     event_timestamp|Amount|Prediction|Fraud_Probability| Status|Risk_Level|
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|2df9d407-8e55-49c...|2026-07-12T07:29:...|  14.8|         0|              0.0|Genuine|       Low|
+--------------------+--------------------+------+----------+-----------------+-------+----------+

Inserted 1 row(s).

Writing Batch 257 to PostgreSQL
-------------------------------------------
Batch: 29
-------------------------------------------
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|      transaction_id|     event_timestamp|Amount|Prediction|Fraud_Probability| Status|Risk_Level|
+--------------------+--------------------+------+--------

-------------------------------------------
Batch: 32
-------------------------------------------
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|      transaction_id|     event_timestamp|Amount|Prediction|Fraud_Probability| Status|Risk_Level|
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|3d70edf0-b343-441...|2026-07-12T07:29:...| 18.95|         0|              0.0|Genuine|       Low|
+--------------------+--------------------+------+----------+-----------------+-------+----------+

Inserted 1 row(s).

Writing Batch 261 to PostgreSQL
-------------------------------------------
Batch: 33
-------------------------------------------
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|      transaction_id|     event_timestamp|Amount|Prediction|Fraud_Probability| Status|Risk_Level|
+--------------------+--------------------+------+--------

-------------------------------------------
Batch: 75
-------------------------------------------
+--------------------+--------------------+-------+----------+-----------------+-------+----------+
|      transaction_id|     event_timestamp| Amount|Prediction|Fraud_Probability| Status|Risk_Level|
+--------------------+--------------------+-------+----------+-----------------+-------+----------+
|d017194f-2902-488...|2026-07-12T07:30:...|1142.02|         0|              0.0|Genuine|       Low|
+--------------------+--------------------+-------+----------+-----------------+-------+----------+

Inserted 1 row(s).

Writing Batch 304 to PostgreSQL


-------------------------------------------
Batch: 76
-------------------------------------------
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|      transaction_id|     event_timestamp|Amount|Prediction|Fraud_Probability| Status|Risk_Level|
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|053f80ff-9608-49b...|2026-07-12T07:30:...| 149.9|         0|              0.0|Genuine|       Low|
+--------------------+--------------------+------+----------+-----------------+-------+----------+

Inserted 1 row(s).

Writing Batch 305 to PostgreSQL
-------------------------------------------
Batch: 77
-------------------------------------------
+--------------------+--------------------+------+----------+-----------------+-------+----------+
|      transaction_id|     event_timestamp|Amount|Prediction|Fraud_Probability| Status|Risk_Level|
+--------------------+--------------------+------+--------

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 